<a href="https://colab.research.google.com/github/k1-r1ll/pytorch/blob/main/15_pytorch_overfitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision.datasets import ImageFolder
from torchvision.datasets import MNIST

In [ ]:
class DigitNN(nn.Module):
  def __init__(self, imp_dim, hid_dim, out_dim):
      super().__init__()
      self.l1 = nn.Linear(imp_dim, hid_dim)
      self.l2 = nn.Linear(hid_dim, out_dim)

  def forward(self, x):
    x = self.l1(x)
    x = nn.functional.relu(x)
    x = self.l2(x)
    return x

In [ ]:
model = DigitNN(28 * 28, 32, 10)

In [ ]:
transforms = tfs.Compose([tfs.ToImage(),
                          tfs.Grayscale(),
                          tfs.ToDtype(torch.float32, scale=True),
                          tfs.Lambda(lambda _img: _img.ravel())])
d_train = ImageFolder("dataset/train", transform=transforms)
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)

In [ ]:
dataset = MNIST('.MNIST', transform=transforms, train=True, download=True)
d_train, d_val = data.random_split(dataset, [0.7, 0.3])
train_data = data.dataLoader(d_train, batch_size=32, shuffle=True)
val_data = data.dataLoader(d_val, batch_size=32, shuffle=False)

In [ ]:
optimizer = optim.Adam(params=model.parameters(), lr=0.01)
loss_func = nn.CrossEntropyLoss()
epochs = 20

In [ ]:
lost_lst_val = []
lost_lst = []

In [ ]:
for _e in range(epochs):
  model.train()
  loss_mean = 0
  lm_count = 0

  train_tqdm = tqdm(train_data, leave=False)

  for x_train, y_train in train_tqdm:
    predict = model(x_train)
    loss = loss_func(predict, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    lm_count += 1

    loss_mean = 1 / lm_count * loss.item() + (1 - 1 / lm_count) * loss_mean
    train_tqdm.set_description(f"Epoch {_e + 1}/{epochs}, loss_mean: {loss_mean}")

  model.eval()
  Q_val = 0
  count_val = 0

  for x_val, y_val in val_data:
    with torch.no_grad():
      p = model(x_val)
      loss = loss_func(p, y_val)
      Q_val += loss.item()
      count_val += 1

  Q_val /= count_val

  lost_lst.append(loss_mean)
  lost_lst_val.append(Q_val)

  print(f"| loss_mean = {loss_mean: .3f}; Q_val = {Q_val: .3f}")

In [ ]:
plt.plot()
plt.plot()
plt.grid()
plt.show()